# Generate the Dead Sea Scrolls documentation
This notebook builds the complete **DSS** example using the current context-first Doc4CF workflow. Run the cells in order from a Doc4CF checkout with a Python 3.13+ kernel, Git, and network access for installation/source download.

Source: [ETCBC/dss](https://github.com/ETCBC/dss), TF version **2.0.4**, pinned revision `3a9ab5b5df4dba72f2179016e8163b61c0a1752d`. The notebook checks out only that revision in its own workspace. Generated files and caches stay in ignored local directories until the optional publication step.

LLM enrichment is enabled by default. Reuse cached dataset/family context and request only missing feature descriptions using **nvidia/nemotron-3.5-super-vl:free**. Set `USE_LLM = False` for a deterministic build without model requests. New model responses may differ from checked-in prose. All measured corpus pages remain based on the pinned dataset.

## 1. Locate and install Doc4CF
Select the Python 3.13+ kernel before running this cell. The editable install uses the code in this checkout; restart the kernel if your notebook environment asks you to.

In [ ]:
from pathlib import Path
import sys
import subprocess
import shutil
import os
import json

assert sys.version_info >= (3, 13), "Select a Python 3.13+ kernel."
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "pyproject.toml").is_file() and (p / "doc4cf").is_dir()), None)
assert ROOT is not None, "Open this notebook inside the Doc4CF checkout."
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(ROOT)], check=True)
assert shutil.which("git"), "Install Git and restart the kernel."

## 2. Fetch the pinned corpus
Only the base TF version is used, without optional annotation modules. A dedicated clone under `.testdata/example-sources` keeps source data separate from generated HTML. Existing clones are reused only if clean; the revision check prevents silent changes to the source.

In [ ]:
NAME = 'DSS'
REVISION = '3a9ab5b5df4dba72f2179016e8163b61c0a1752d'
VERSION = '2.0.4'
SOURCE = ROOT / ".testdata" / "example-sources" / NAME
SOURCE.parent.mkdir(parents=True, exist_ok=True)
if not SOURCE.exists():
    subprocess.run(["git", "clone", "--filter=blob:none",
                    "https://github.com/ETCBC/dss.git", str(SOURCE)], check=True)
status = subprocess.check_output(["git", "-C", str(SOURCE), "status", "--porcelain"], text=True)
source_changes = [
    line for line in status.splitlines()
    if not line.startswith(f"?? tf/{VERSION}/.cfm/")
]
assert not source_changes, "Source checkout has local changes: " + "; ".join(source_changes)
subprocess.run(["git", "-C", str(SOURCE), "fetch", "origin", REVISION], check=True)
subprocess.run(["git", "-C", str(SOURCE), "checkout", "--detach", REVISION], check=True)
assert subprocess.check_output(["git", "-C", str(SOURCE), "rev-parse", "HEAD"], text=True).strip() == REVISION
CORPUS = SOURCE / "tf" / VERSION
assert len(list(CORPUS.glob("*.tf"))) == 79
assert (CORPUS / "otype.tf").is_file() and (CORPUS / "oslots.tf").is_file()

## 3. Configure context analysis and enrichment

`USE_LLM = True` resumes the context-first enrichment workflow, reusing successful responses from the local cache. Set it to `False` for measured documentation without model calls. Set `TARGETED_EVIDENCE = True` only if you want a planning request to retrieve rare values or examples before each feature description; this adds model requests. The context pages and review report are enabled for enriched builds. `MARKDOWN = True` exports the same descriptions without additional requests.

The OpenRouter key is read from `OPENROUTER_API_KEY` or entered through a hidden prompt. It is never included in a command or notebook output. The editorial notes path is configured below. This corpus has no additional editorial context by default. Set CONTEXT to a UTF-8 file if you want editorial notes included during feature generation.

Review the dataset, family, and relationship artifacts before generating feature descriptions or publishing the site.

In [ ]:
USE_LLM = True
TARGETED_EVIDENCE = False
CONTEXT_PAGES = True
MARKDOWN = True
PROVIDER = "openrouter"
DATASET_MODEL = 'nvidia/nemotron-3.5-super-vl:free'
DATASET_INSTRUCTIONS = None  # Optionally set a Path to UTF-8 dataset instructions.
FAMILY_MODEL = 'nvidia/nemotron-3.5-super-vl:free'
FEATURE_MODEL = 'nvidia/nemotron-3.5-super-vl:free'
CONTEXT = None
CONTEXT_DIR = ROOT / ".testdata" / "context-artifacts" / 'DSS'
OUTPUT = ROOT / ".testdata" / "example-builds" / 'DSS' / ("enriched" if USE_LLM else "offline")
CACHE = ROOT / ".doc4cf-cache" / 'dss'

if USE_LLM:
    from getpass import getpass
    if not os.environ.get("OPENROUTER_API_KEY"):
        os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter API key: ")
    assert os.environ.get("OPENROUTER_API_KEY"), "An API key is required for enrichment."

## 4. Analyze the dataset and inspect context

This standalone step writes `inventory.json`, `dataset_context.json`, `feature_families.json`, and `feature_relationships.json`. Inventory export automatically harvests the sibling app/ directory structured YAML and AST summaries of Python behavior, without executing app code. Set DATASET_INSTRUCTIONS to an optional UTF-8 file to guide dataset-level focus and phrasing; it cannot override evidence rules, is included in the cache key, and its hash is recorded. Context analysis uses separate dataset and family model settings, is cached, and does not build the site. If a request is interrupted, rerun this cell after the provider is available; completed responses are reused.

Before feature generation, measured editorial clarifications distinguish populated occurrence links from scalar edge values and remove an unsupported future-date inference. The raw model cache is preserved; the added conventions are labeled editorial.


In [ ]:
if USE_LLM:
    analysis_command = [
        sys.executable, "-m", "doc4cf", str(CORPUS),
        "--analyze-context", str(CONTEXT_DIR),
        "--provider", PROVIDER,
        "--context-model", DATASET_MODEL,
        "--family-model", FAMILY_MODEL,
        "--cache", str(CACHE),
    ]
    if DATASET_INSTRUCTIONS is not None:
        analysis_command += ["--dataset-instructions", str(DATASET_INSTRUCTIONS)]
    analysis = subprocess.run(analysis_command, cwd=ROOT, capture_output=True,
                             text=True, encoding="utf-8", errors="replace")
    print(analysis.stdout)
    print(analysis.stderr)
    if analysis.returncode:
        raise RuntimeError("Context analysis failed; review the error and rerun this cell.")
    subprocess.run([sys.executable, str(ROOT / "scripts" / "apply_dss_context_notes.py"),
                    str(CONTEXT_DIR)], cwd=ROOT, check=True)
    for artifact in ("inventory.json", "dataset_context.json", "feature_families.json", "feature_relationships.json"):
        assert (CONTEXT_DIR / artifact).is_file(), artifact
    print("Context artifacts:", CONTEXT_DIR)
else:
    print("Context analysis skipped because USE_LLM is False.")

### Review the generated artifacts

Inspect the dataset interpretation, feature families, relationship statuses, evidence paths, and `needs_review` fields. These files are the reusable model context for the next step. If you revise them, retain the artifact structure and inventory hash so Doc4CF can validate them.

In [ ]:
if USE_LLM:
    for artifact in ("dataset_context.json", "feature_families.json", "feature_relationships.json"):
        print("\n###", artifact)
        print((CONTEXT_DIR / artifact).read_text(encoding="utf-8"))

## 5. Generate the site with reviewed context

Prepare feature descriptions sequentially, reusing successful cached responses, then render the HTML and Markdown site once. Extra evidence-planning calls remain opt-in. If the daily free quota is exhausted, the build includes completed descriptions and reports remaining features separately; rerun after the quota resets. Review the generated prose and `context_review.json` before publication.

After rendering, source-cited clarifications keep feature provenance, editorial authorship, value semantics, and text-format use grounded in the pinned sources. Edited prose is labeled Reviewed description; the raw model cache remains unchanged.

For the same corpus fingerprint and measured facts, the existing example supplies its complete association tables and rich Markdown reference. New AI descriptions are added without discarding those measurements.


In [ ]:
base_command = [sys.executable, "-m", "doc4cf", str(CORPUS), str(OUTPUT),
                "--project-home", "../../index.html"]
if MARKDOWN:
    base_command.append("--markdown")
if USE_LLM:
    base_command += ["--provider", PROVIDER, "--model", FEATURE_MODEL,
                     "--feature-model", FEATURE_MODEL,
                     "--context-dir", str(CONTEXT_DIR), "--cache", str(CACHE)]
    if CONTEXT_PAGES:
        base_command.append("--context-pages")
    if TARGETED_EVIDENCE:
        base_command.append("--targeted-evidence")
    if CONTEXT is not None:
        CONTEXT = Path(CONTEXT)
        assert CONTEXT.is_file(), CONTEXT
        base_command += ["--dataset-context", str(CONTEXT)]

completed_features = []
quota_exhausted = False
if USE_LLM:
    from doc4cf.context import contextual_description_record, load_context_artifacts
    from doc4cf.generator import (
        _feature_description_evidence, _observed_values, dataset_inventory,
        header, load_corpus, profile,
    )

    OUTPUT.mkdir(parents=True, exist_ok=True)
    files = {path.stem: header(path) for path in sorted(CORPUS.glob("*.tf"))}
    api = load_corpus(CORPUS, files, OUTPUT)
    assert api, "Context-Fabric could not load the corpus."
    description_inventory = dataset_inventory(CORPUS, files, api)
    description_context = load_context_artifacts(CONTEXT_DIR, description_inventory)
    observed_values = _observed_values(api, files)
    order = api.F.otype.all
    node_counts = {node_type: len(api.F.otype.s(node_type)) for node_type in order}
    editorial_notes = CONTEXT.read_text(encoding="utf-8-sig") if CONTEXT is not None else ""
    for index, (feature_name, info) in enumerate(files.items(), start=1):
        measured = profile(api, feature_name, info, order, node_counts)
        evidence = _feature_description_evidence(
            description_inventory["corpus_label"], measured, info["meta"],
            description_inventory.get("application"), editorial_notes, description_context,
        )
        try:
            contextual_description_record(
                evidence, FEATURE_MODEL, CACHE, PROVIDER, set(files), observed_values,
                api=api, feature_kind=info["kind"], targeted_evidence=TARGETED_EVIDENCE,
            )
        except RuntimeError as error:
            if "OpenRouter daily free-model quota exhausted" not in str(error):
                raise
            print(f"OpenRouter free quota exhausted at feature {feature_name}.")
            print(f"Keeping {len(completed_features)} completed feature description(s).")
            quota_exhausted = True
            break
        completed_features.append(feature_name)
        print(f"Description ready {index}/{len(files)}: {feature_name}", flush=True)

    if completed_features:
        final_command = base_command + ["--describe-features", ",".join(completed_features)]
    else:
        final_command = [sys.executable, "-m", "doc4cf", str(CORPUS), str(OUTPUT),
                         "--project-home", "../../index.html",
                         "--cache", str(CACHE)]
        if CONTEXT_PAGES:
            final_command += ["--context-dir", str(CONTEXT_DIR), "--context-pages"]
        if MARKDOWN:
            final_command.append("--markdown")
else:
    final_command = base_command

result = subprocess.run(final_command, cwd=ROOT, capture_output=True, text=True,
                        encoding="utf-8", errors="replace")
print("EXIT CODE:", result.returncode)
if result.returncode:
    print("\n--- STDOUT ---")
    print(result.stdout)
    print("\n--- STDERR ---")
    print(result.stderr)
    raise RuntimeError("Documentation generation failed; do not publish its partial output.")
shutil.copyfile(SOURCE / 'LICENSE', OUTPUT / 'DATA-LICENSE.txt')
if USE_LLM:
    subprocess.run([sys.executable, str(ROOT / "scripts" / "apply_dss_context_notes.py"),
                    str(OUTPUT), "site", str(ROOT / "examples" / "DSS")], cwd=ROOT, check=True)
print("Generated:", OUTPUT / "index.html")
if quota_exhausted:
    print("Completed cached inferences are included; remaining features keep their measured descriptions.")


## 6. Check completeness and inspect the pages

Verify every HTML and Markdown feature page, graph identifiers, overview, audit, and optional context pages. Report generated-description coverage separately from page coverage. Inspect sample descriptions and the context review flags before publication.


In [ ]:
features = {p.stem for p in CORPUS.glob("*.tf")}
assert {p.stem for p in (OUTPUT / "features").glob("*.html")} == features
graph = json.loads((OUTPUT / "knowledge_graph.json").read_text(encoding="utf-8"))
assert {"feature::" + f for f in features} <= graph["nodes"].keys()
required_pages = ["index.html", "features.html", "structure.html", "diagram.html", "text.html", "audit.html"]
if USE_LLM and CONTEXT_PAGES:
    required_pages += ["dataset.html", "relationships.html", "context_review.json", "context_manifest.json"]
for page in required_pages:
    assert (OUTPUT / page).is_file(), page
for name in features:
    html = (OUTPUT / "features" / (name + ".html")).read_text(encoding="utf-8")
    if USE_LLM and name in completed_features:
        assert 'class="inferred-description"' in html
if MARKDOWN:
    markdown_output = OUTPUT / "markdown"
    assert (markdown_output / "index.md").is_file()
    assert (markdown_output / "features.md").is_file()
    assert {p.stem for p in (markdown_output / "features").glob("*.md")} == features
    print(f"Verified {len(features)} Markdown feature pages and both indexes.")
print(f"Verified {len(features)} feature pages and {len(required_pages)} overview/review pages.")
print(f"Generated descriptions: {len(completed_features)}/{len(features)}.")
print((OUTPUT / "index.html").as_uri())


## 7. Optional: replace the checked-in example

Set `PUBLISH = True` only after reviewing the generated pages and context review report. This copies the build into `examples/<corpus>/`; it makes no Git commit or push. An offline build replaces enriched feature descriptions with corpus-only pages. The Pages workflow publishes checked-in files without calling an LLM.

Inspect `git diff` before committing. Notebook outputs are not cleared automatically. Before sharing or committing, run `python scripts/clear_notebook_outputs.py`. CI checks this after each push and on pull requests; it fails when outputs or execution counts remain. GitHub Actions cannot clean the file before it is pushed; a required check can block a pull request merge, but a direct push has already reached GitHub. Keep credentials, context caches, and downloaded corpora out of Git.

In [ ]:
PUBLISH = False
if PUBLISH:
    destination = ROOT / "examples" / NAME
    shutil.copytree(OUTPUT, destination, dirs_exist_ok=True)
    print("Updated:", destination)
else:
    print("Review build retained locally; checked-in example unchanged.")